# Collusion sprint: run

**Run all** does: setup → canary → context probe → launch checks → pilot → analysis. It stops at the first
failure, so a failed setup, canary or probe never reaches the pilot.

- The OpenRouter key comes from Colab Secrets (`OPENROUTER_API_KEY`, notebook access on). It is never printed or saved.
- Journals and call logs go straight to Drive. If the runtime dies, Run all again: finished games are skipped and interrupted ones resume.
- `--budget` is the total number of API attempts (retries included) a run directory may use. Attempts already in its call logs count, so rerunning never grants a fresh allowance. Don't run one family from two notebooks at once.

In [ ]:
# Setup: code at a fixed commit. SOURCE is a git URL or a .bundle file on Drive (git clone accepts both).
SOURCE = "/content/drive/MyDrive/collusion-sprint/collusion-sprint.bundle"
COMMIT = "4c9880ad35bfde677853cdf3dd47dbc6ab64f61b"  # the code commit under review
REPO = "/content/collusion-sprint"

import os, signal, subprocess, sys
from google.colab import drive

def sh(cmd):
    """Run a shell command with live output; raise (stopping Run all) if it fails."""
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="")
    except KeyboardInterrupt:  # pass the interrupt on, so games stop cleanly and keep their journals
        proc.send_signal(signal.SIGINT)
        proc.wait()
        raise
    if proc.wait():
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {cmd}")

drive.mount("/content/drive")
sh(f"rm -rf {REPO} && git clone -q {SOURCE} {REPO}")
os.chdir(REPO)
sys.path.insert(0, REPO)
sh(f"git checkout -q {COMMIT} && git log -1 --format='%H %s'")
sh("bash scripts/get_starter.sh")
sh("pip install -q -e . -e starter/util -e starter/algorithmic-pricing-llms")

In [ ]:
# Key, output folder, model family.
from google.colab import userdata
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")  # never print it
OUT = "/content/drive/MyDrive/collusion-sprint/runs"
FAMILY = "deepseek"  # deepseek | qwen | mistral
RUN = f"python -m sprint.run --out {OUT}"

## Canary: one tiny game, about 41 decision calls

Integration only: it is not a main-study game and never pools with one.

In [ ]:
sh(f"{RUN} configs/canary_{FAMILY}.toml --budget 100")

## Context probe: one call

A full 100-round window of three sellers, with realistic plans and insights each about `max_tokens` tokens long
(together about twice what one reply can hold). It checks that a long prompt fits, gets answered and parses.
It is a probe, not a proof of the longest possible prompt.

In [ ]:
sh(f"{RUN} configs/main_pilot_{FAMILY}.toml --budget 10 --context-probe")

## Launch checks: all must pass, or Run all stops here

In [ ]:
from sprint import analyze
checks = analyze.launch_checks(f"{OUT}/canary_{FAMILY}", f"{OUT}/main_{FAMILY}_probe")
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", name)
assert all(checks.values()), "launch checks failed: not starting the pilot"

## Pilot: 4 main-study games, up to 2,840 decision calls before retries

In [ ]:
sh(f"{RUN} configs/main_pilot_{FAMILY}.toml --budget 3500 --dry-run")
sh(f"{RUN} configs/main_pilot_{FAMILY}.toml --budget 3500")

## Analysis

In [ ]:
import glob
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)
table = analyze.games([f"{OUT}/main_{FAMILY}"])
display(table)
display(analyze.summary(table))
analyze.plot_gains(table)
for path in sorted(glob.glob(f"{OUT}/main_{FAMILY}/game_*.jsonl")):
    if not path.endswith(".calls.jsonl"):
        analyze.plot_prices(path).set_title(path.split("/")[-1])
        display(analyze.rival_responses(path))